# Natural Language Processing (NLP)

This notebook covers the essentials of Natural Language Processing (NLP), starting from text preprocessing, proceeding to numeric text representations, and finishing with pre-trained Transformer models from Hugging Face.

---

## 1. Text Preprocessing

Raw text data is unstructured and noisy. To prepare text for machine learning models, we clean and normalize it through the following steps:
- **Lowercasing**: Converting all text to lowercase to ensure consistency (e.g., "Apple" vs. "apple").
- **Tokenization**: Splitting sentences into individual words (tokens).
- **Stopwords Removal**: Removing common words that carry little semantic meaning (e.g., "the", "is", "and").
- **Stemming**: Reducing words to their base form by chopping off prefixes/suffixes (e.g., "running" $\rightarrow$ "run"). Uses heuristic rules.
- **Lemmatization**: Reducing words to their vocabulary dictionary form (lemma) using linguistic rules (e.g., "better" $\rightarrow$ "good").

In [1]:
# Install NLTK if not already installed
# !pip install nltk

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import string

# Download NLTK datasets needed for tokenization and lemmatization
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

raw_text = "Natural Language Processing (NLP) is exciting! Learning models are training rapidly."

# 1. Convert to lowercase
text_lower = raw_text.lower()

# 2. Tokenization
tokens = word_tokenize(text_lower)
print(f"Tokens:\n{tokens}\n")

# 3. Remove stopwords & punctuation
stop_words = set(stopwords.words('english'))
filtered_tokens = [w for w in tokens if w not in stop_words and w not in string.punctuation]
print(f"Filtered Tokens:\n{filtered_tokens}\n")

# 4. Stemming vs Lemmatization
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = [stemmer.stem(w) for w in filtered_tokens]
lemmatized_words = [lemmatizer.lemmatize(w) for w in filtered_tokens]

print(f"Stemmed Words:\n{stemmed_words}\n")
print(f"Lemmatized Words:\n{lemmatized_words}")

Tokens:
['natural', 'language', 'processing', '(', 'nlp', ')', 'is', 'exciting', '!', 'learning', 'models', 'are', 'training', 'rapidly', '.']

Filtered Tokens:
['natural', 'language', 'processing', 'nlp', 'exciting', 'learning', 'models', 'training', 'rapidly']



Stemmed Words:
['natur', 'languag', 'process', 'nlp', 'excit', 'learn', 'model', 'train', 'rapidli']

Lemmatized Words:
['natural', 'language', 'processing', 'nlp', 'exciting', 'learning', 'model', 'training', 'rapidly']


## 2. Text Representation (Vectorization)

Machine learning models require numerical input. We convert cleaned text into numerical matrices using representations like:

### 2.1 Bag-of-Words (BoW)
A representation that models the count of words in each document, ignoring order and grammar.

### 2.2 TF-IDF (Term Frequency-Inverse Document Frequency)
A statistical metric that weights words based on how unique they are to specific documents across a corpus:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$
Where:
- $\text{TF}(t, d) = \frac{\text{Count of term } t \text{ in document } d}{\text{Total terms in document } d}$
- $\text{IDF}(t, D) = \log\left(\frac{\text{Total number of documents } D}{1 + \text{Number of documents containing term } t}\right)$

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Sample text corpus
corpus = [
    "I love data science and machine learning.",
    "Machine learning models are powered by mathematics and statistics.",
    "Data science includes text analysis and natural language processing."
]

# Initialize and compute TF-IDF matrix
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

# Convert to DataFrame for visualization
feature_names = tfidf_vectorizer.get_feature_names_out()
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

print("TF-IDF Feature Columns:")
print(list(feature_names))
print("\nTF-IDF Matrix Representation:")
print(df_tfidf.round(3))

TF-IDF Feature Columns:
['analysis', 'and', 'are', 'by', 'data', 'includes', 'language', 'learning', 'love', 'machine', 'mathematics', 'models', 'natural', 'powered', 'processing', 'science', 'statistics', 'text']

TF-IDF Matrix Representation:
   analysis    and    are     by   data  includes  language  learning   love  \
0     0.000  0.309  0.000  0.000  0.397     0.000     0.000     0.397  0.523   
1     0.000  0.216  0.365  0.365  0.000     0.000     0.000     0.278  0.000   
2     0.365  0.216  0.000  0.000  0.278     0.365     0.365     0.000  0.000   

   machine  mathematics  models  natural  powered  processing  science  \
0    0.397        0.000   0.000    0.000    0.000       0.000    0.397   
1    0.278        0.365   0.365    0.000    0.365       0.000    0.000   
2    0.000        0.000   0.000    0.365    0.000       0.365    0.278   

   statistics   text  
0       0.000  0.000  
1       0.365  0.000  
2       0.000  0.365  


## 3. Word Embeddings

Traditional BoW/TF-IDF models create sparse matrices and ignore word meaning/relationships. **Word Embeddings** solve this by mapping words to dense vector spaces where semantically similar words are physically closer to each other.

- **Word2Vec**: Learns embeddings using shallow neural networks. Has two training approaches:
  1. **CBOW (Continuous Bag-of-Words)**: Predicts a target word from context words.
  2. **Skip-gram**: Predicts context words from a target word.
- **Vector Math**: Embeddings represent relationships such that $\text{vector('king')} - \text{vector('man')} + \text{vector('woman')} \approx \text{vector('queen')}$.

In [3]:
# Install gensim if not already installed
# !pip install gensim

from gensim.models import Word2Vec

# Small tokenized reviews corpus for embeddings training
restaurant_reviews = [
    ["the", "food", "was", "amazing", "and", "delicious"],
    ["terrible", "service", "and", "cold", "food"],
    ["amazing", "ambiance", "great", "staff", "service"],
    ["worst", "experience", "ever", "terrible", "food"],
    ["the", "service", "was", "slow", "but", "food", "was", "ok"]
]

# Train a Word2Vec Skip-gram model (sg=1)
w2v_model = Word2Vec(sentences=restaurant_reviews, vector_size=32, window=3, min_count=1, sg=1, epochs=15)

# Get vector representation of a word
food_vector = w2v_model.wv["food"]
print(f"Vector shape of 'food': {food_vector.shape}")
print(f"First 5 components of 'food' vector: {food_vector[:5]}\n")

# Find words semantically similar to 'food'
similar_words = w2v_model.wv.most_similar("food", topn=3)
print(f"Most similar words to 'food':\n{similar_words}")

Vector shape of 'food': (32,)
First 5 components of 'food' vector: [-0.00166436  0.00070933  0.01601301  0.02821896 -0.02912705]

Most similar words to 'food':
[('worst', 0.36920326948165894), ('ever', 0.36896029114723206), ('ambiance', 0.21754460036754608)]


## 4. Sentiment Classification

Now let's build a practical end-to-end classifier that vectorizes restaurant reviews and classifies them as Positive (1) or Negative (0).

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Custom reviews dataset
reviews_dataset = {
    "review": [
        "The steak was incredibly delicious and tender.",
        "Amazing experience, helpful staff, and top-notch drinks.",
        "Highly recommended, will definitely visit again!",
        "The service was terrible and the waiter was extremely rude.",
        "Cold soup and sticky tables, won't return.",
        "Extremely overpriced for such poor food quality."
    ],
    "sentiment": [1, 1, 1, 0, 0, 0]  # 1: Positive, 0: Negative
}

df = pd.DataFrame(reviews_dataset)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df["review"], df["sentiment"], test_size=0.33, random_state=42)

# Convert reviews to TF-IDF representations
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train classifier
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

# Predict on test set
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.1f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

# Test on custom query
custom_query = ["The ambiance was lovely and the dessert was fantastic."]
custom_vec = vectorizer.transform(custom_query)
prediction = model.predict(custom_vec)[0]
print(f"Query: '{custom_query[0]}'")
print(f"Prediction: {'Positive' if prediction == 1 else 'Negative'}")

Model Accuracy: 0.0%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       0.0
    Positive       0.00      0.00      0.00       2.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0

Query: 'The ambiance was lovely and the dessert was fantastic.'
Prediction: Negative


C:\Users\blgnr\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\blgnr\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\blgnr\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

## 5. Transformers & Hugging Face

Modern NLP has shifted from training models from scratch to fine-tuning massive pre-trained architectures (like BERT, GPT, RoBERTa) based on the **Attention Mechanism**.

Hugging Face's `transformers` library provides pre-trained models that can perform complex NLP tasks with minimal code.

In [5]:
# To run pre-trained transformers, install:
# !pip install transformers torch

print("Hugging Face Pipeline Code Stub (Uncomment inside to run):\n")
print("""
from transformers import pipeline

# 1. Initialize Sentiment Classifier Pipeline
classifier = pipeline("sentiment-analysis")
result = classifier("This restaurant has the best steak in town!")
print("Sentiment Analysis Result:", result)

# 2. Initialize Text Generation Pipeline
generator = pipeline("text-generation", model="gpt2")
generated_text = generator("Artificial Intelligence in NLP is", max_length=15, num_return_sequences=1)
print("\nGenerated Text:", generated_text[0]['generated_text'])
""")

Hugging Face Pipeline Code Stub (Uncomment inside to run):


from transformers import pipeline

# 1. Initialize Sentiment Classifier Pipeline
classifier = pipeline("sentiment-analysis")
result = classifier("This restaurant has the best steak in town!")
print("Sentiment Analysis Result:", result)

# 2. Initialize Text Generation Pipeline
generator = pipeline("text-generation", model="gpt2")
generated_text = generator("Artificial Intelligence in NLP is", max_length=15, num_return_sequences=1)
print("
Generated Text:", generated_text[0]['generated_text'])

